# 05 Build the drug panel

Notebooks 02 and 03 built the two models. This notebook builds the **drug side** of the screen: conditions 2 and 3, which describe the drug rather than the plant compound.

| Condition | Question | Where the answer comes from |
| --- | --- | --- |
| **2** | Is this drug cleared by CYP3A4? | lab measurements, 670 drugs |
| **3** | Does this drug cause heart trouble when it builds up? | lab measurements where they exist, Model B otherwise |

Condition 2 is **never** predicted. If a drug is not in the data, this notebook says so rather than guessing, because inventing a link in the chain is how a screen starts producing confident nonsense.

## Why these datasets and not the FDA and CredibleMeds ones

The original plan was the FDA substrate table crossed with the CredibleMeds QT drug list. Neither one works, for different reasons:

- **The FDA table** is a web page with no download, and its CYP3A row lists only **two drugs**: midazolam and triazolam. Those are *reference* drugs used when designing an interaction study, not a list of everything CYP3A4 clears. A screen built on it would cover two drugs.
- **The CredibleMeds list** is free to read after signing up, but downloading it requires a paid licence, so it cannot be included in this project.

Both answers are available as lab measurements from TDC instead, downloaded back in notebook 01 from the same place as the training data.

## What this costs us, stated plainly

CredibleMeds records what actually happened to **patients**. The hERG dataset records what happens to a channel in a **dish**. Those are not the same claim.

So the output of this screen is a list of combinations worth testing in a lab, not a list of combinations known to harm people. That is a narrower claim than the proposal first made, and it is the one the available data actually supports.

In exchange, every hand-made file is gone: the whole screen now runs from scratch with no sign-ups and no structures typed in by hand.

In [1]:
import sys
import importlib
import os

sys.path.append(os.path.abspath(os.path.join("..", "src")))

import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

import names
import combine

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")


# Python only reads a file from src/ the FIRST time it is imported. After that
# it serves a cached copy, so any edit you make to src/ is invisible to a
# notebook that has already run this cell.
#
# The symptom is confusing: an AttributeError saying a function does not exist,
# when you can see it sitting right there in the file. Reloading here means
# "Run All" always uses what is actually on disk.
for module in (names, combine):
    importlib.reload(module)

## Load the two lab datasets

Both were downloaded in notebook 01. Neither contains any prediction: every row is something a lab actually measured.

In [2]:
SUBSTRATE_PATH = os.path.join(RAW_DIR, "cyp3a4_substrate_carbonmangels.tab")
HERG_PATH = os.path.join(RAW_DIR, "herg.tab")

for path in (SUBSTRATE_PATH, HERG_PATH):
    if not os.path.exists(path):
        raise SystemExit(f"{os.path.basename(path)} is missing. Run notebook 01 first.")

substrate_df = pd.read_csv(SUBSTRATE_PATH, sep="\t")
herg_df = pd.read_csv(HERG_PATH, sep="\t")

print(f"CYP3A4 substrate set : {len(substrate_df)} drugs")
print(f"   substrates        : {int((substrate_df['Y'] == 1).sum())}")
print(f"   non-substrates    : {int((substrate_df['Y'] == 0).sum())}")
print()
print(f"hERG set             : {len(herg_df)} compounds")
print(f"   blockers          : {int((herg_df['Y'] == 1).sum())}")

CYP3A4 substrate set : 670 drugs
   substrates        : 355
   non-substrates    : 315

hERG set             : 655 compounds
   blockers          : 451


## Match the two sets by structure, not by name

We need to know which drugs appear in **both** datasets. The obvious way is to match on the drug name, and it does not work.

The two files name things completely differently. The substrate set uses plain generic names (`amiodarone`). The hERG set uses whatever its source paper used (`DEMETHYLASTEMIZOLE`, `GBR-12909`). Match on those strings and we would miss most of the real overlap.

**So match on the structure instead.** The catch is that the same molecule can be written as several different valid SMILES strings, so comparing the raw text does not work either. RDKit can rewrite any molecule into one agreed standard form — its *canonical* SMILES. Do that to both sides and the same molecule lands on the same key no matter what it was called or how its structure was written.

In [3]:
def canonical(smiles):
    # Rewrite every molecule into one standard form, so the same molecule from
    # two different sources produces the same text. Comparing the raw SMILES
    # instead would miss most of the matches.
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol) if mol is not None else None


for frame in (substrate_df, herg_df):
    frame["canonical"] = frame["Drug"].map(canonical)

substrates = substrate_df[(substrate_df["Y"] == 1) & substrate_df["canonical"].notna()]
herg_ok = herg_df[herg_df["canonical"].notna()]
herg_blockers = set(herg_ok.loc[herg_ok["Y"] == 1, "canonical"])
herg_measured = set(herg_ok["canonical"])

print(f"CYP3A4 substrates with a usable structure : {len(substrates)}")
print(f"   also measured in the hERG set          : "
      f"{int(substrates['canonical'].isin(herg_measured).sum())}")
print(f"   measured hERG BLOCKERS                 : "
      f"{int(substrates['canonical'].isin(herg_blockers).sum())}")

CYP3A4 substrates with a usable structure : 355
   also measured in the hERG set          : 36
   measured hERG BLOCKERS                 : 23


## Build the drug panel

Now assemble the list of drugs the screen will actually check.

**Drugs measured as hERG NON-blockers are dropped.** Condition 3 is a measured *no* for them. That breaks the chain, so no pair involving them could ever be flagged, and keeping them would only generate rows that get thrown away later.

**Everything else splits into two groups**, which is exactly the order of preference already built into `combine.drug_blocks_herg`:

| Group | Condition 3 comes from | How strong |
| --- | --- | --- |
| has a measured hERG result | the lab measurement | firm yes, beats any probability |
| no measurement exists | Model B's prediction | depends on the probability |

A lab result always wins over a prediction. Model B never gets asked about a drug where real data exists.

In [4]:
panel = substrates[~substrates["canonical"].isin(herg_measured - herg_blockers)].copy()
panel["name"] = panel["Drug_ID"].astype(str)
panel["smiles"] = panel["Drug"]
panel["strength"] = "measured substrate (Carbon-Mangels)"
panel["herg_measured"] = panel["canonical"].isin(herg_blockers)

n_measured = int(panel["herg_measured"].sum())
print(f"drug panel: {len(panel)} measured CYP3A4 substrates")
print(f"   condition 3 from measured hERG blockade : {n_measured}")
print(f"   condition 3 left to Model B             : {len(panel) - n_measured}")
print(f"   dropped as measured hERG non-blockers   : {len(substrates) - len(panel)}")
print()
print("the measured blockers, which carry the strongest condition 3:")
print("  " + ", ".join(sorted(panel.loc[panel["herg_measured"], "name"])))

drug panel: 342 measured CYP3A4 substrates
   condition 3 from measured hERG blockade : 23
   condition 3 left to Model B             : 319
   dropped as measured hERG non-blockers   : 13

the measured blockers, which carry the strongest condition 3:
  (+)-tolterodine, amiodarone, amitriptyline, amoxapine, aripiprazole, buspirone, caffeine, celecoxib, clomipramine, clotrimazole, digitoxin, disopyramide, gallopamil, imipramine, loratadine, mianserin, miconazole, ritonavir, sertindole, tomoxetine, trazodone, vanoxerine, vesnarinone


## Save the panel

One file, `screen_drugs.csv`, carrying the drug names, their structures, and both conditions as columns. Notebook 06 reads it and does not need to know where any of it came from.

In [5]:
panel["key"] = panel["name"].map(names.normalize_drug_name)

# Catch two rows that clean up to the same name. Left alone they would each
# pair with every jamu compound, silently doubling those rows in the final
# screen and making one drug look like two separate findings.
dupes = panel[panel["key"].duplicated(keep=False) & (panel["key"] != "")]
if len(dupes):
    print("duplicate normalised names:")
    print(dupes[["name", "key"]].sort_values("key").to_string(index=False))
    panel = panel.drop_duplicates("key").reset_index(drop=True)
    print(f"   kept the first of each, {len(panel)} drugs remain")
    print()

# One file, not three. This used to be three separate CSVs that had to be kept
# in step by hand. Conditions 2 and 3 are just two columns of this one table,
# and combine.lookups_from_panel builds both lookups from it, so there is no
# way for them to drift out of agreement.
panel[["name", "smiles", "strength", "herg_measured", "key"]].to_csv(
    os.path.join(PROCESSED_DIR, "screen_drugs.csv"), index=False)

# Build the lookups once right here, purely as a check. A blank name or a
# duplicate key then shows up now, instead of halfway through the screen.
substrate_lookup, herg_lookup = combine.lookups_from_panel(panel)
print(f"substrate lookup : {len(substrate_lookup)} keys")
print(f"hERG lookup      : {len(herg_lookup)} keys")
print()
print(f"saved screen_drugs.csv, {len(panel)} drugs")

substrate lookup : 342 keys
hERG lookup      : 23 keys

saved screen_drugs.csv, 342 drugs


## What this panel can and cannot support

- **Condition 2 is a measurement, not a prediction.** A drug is in this panel because a lab tested it and found CYP3A4 clears it.

- **Condition 3 is a measurement for some drugs and a prediction for the rest.** The `cond3_source` column records which, on every single row, so the two kinds of evidence never get quietly mixed together when results are quoted.

- **The hERG label is a lab assay, not a patient outcome.** It says the molecule blocks the channel in a dish. It does not say the drug causes arrhythmia in people at normal doses. The original plan used clinical risk categories, which are a stronger statement, but that list cannot be redistributed.

- **Not being in the panel means no measurement was found. It never means a drug is safe.**